In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd


In [2]:
def scrape_faq_page(url):
    resp = requests.get(url)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    faqs = []

    # Very simple pattern: questions in <h2> or <h3>, answer is the next <p> or <div>
    for heading in soup.find_all(["h2", "h3"]):
        q_text = heading.get_text(strip=True)
        if not q_text:
            continue

        # Grab all text until the next heading of same level
        answer_parts = []
        for sib in heading.find_next_siblings():
            if sib.name in ["h2", "h3"]:
                break
            answer_parts.append(sib.get_text(" ", strip=True))

        a_text = " ".join(part for part in answer_parts if part)
        if a_text:
            faqs.append({"question": q_text, "answer": a_text})

    return faqs

url = "https://www.douglascollege.ca/student-services/student-resources/covid19/international-faqs"
faqs = scrape_faq_page(url)
len(faqs), faqs[:3]  # show a few


(14,
 [{'question': 'Am I allowed to travel to Canada?',
   'answer': 'Travel exemptions and restrictions are constantly changing. Please visit our Travelling to Canada page and the Immigration, Refugees, and Citizenship Canada (IRCC) website for the most up-to-date information.'},
  {'question': 'I have not yet arrived in Canada, can I start classes online in my country?',
   'answer': 'The majority of courses in the Winter 2022 semester at Douglas College are currently scheduled to be in-person. Classes that are scheduled as “in-person” or “hybrid” require students to be here from the beginning of the semester. If none of your classes have in-class components that required you to be on campus and you choose to study online from your country, please review these considerations: Some courses have scheduled times and students are expected to be online in the sessions Please account for time zones when you select courses You will be required to stream videos and video conferencing You ar

In [3]:
faq_urls = [
    "https://www.douglascollege.ca/student-services/student-resources/covid19/international-faqs",
    "https://www.douglascollege.ca/future-students/apply-douglas/domestic-students/admissions-faq",
    "https://www.douglascollege.ca/current-students/important-dates-information/grading-faq",
    "https://www.douglascollege.ca/current-students/enrolment-services/fees-related-information/fees-faq",
    "https://www.douglascollege.ca/current-students/advising-services/advising-services/advising-services-faq",
    "https://www.douglascollege.ca/student-services/student-support/counselling/counselling-faq",
    "https://www.douglascollege.ca/student-services/student-resources/covid19/academic-faqs",
]

all_faqs = []
for url in faq_urls:
    page_faqs = scrape_faq_page(url)
    for qa in page_faqs:
        qa["source_url"] = url
        all_faqs.append(qa)

df = pd.DataFrame(all_faqs)
df.insert(0, "id", range(1, len(df) + 1))
df.head()


,id,question,answer,source_url
0,1,Am I allowed to travel to Canada?,Travel exemptions and restrictions are constan...,https://www.douglascollege.ca/student-services...
1,2,"I have not yet arrived in Canada, can I start ...",The majority of courses in the Winter 2022 sem...,https://www.douglascollege.ca/student-services...
2,3,Do I need a study permit to take classes if th...,Yes. You are required to have a valid study pe...,https://www.douglascollege.ca/student-services...
3,4,How will my PGWP eligibility be impacted by CO...,Students may count all their time studying onl...,https://www.douglascollege.ca/student-services...
4,5,"I am out of the country, when should I come to...",Classes that are scheduled as “in-person” or “...,https://www.douglascollege.ca/student-services...


In [4]:
df.to_csv("douglas_faqs.csv", index=False)

In [5]:
faq_df = pd.read_csv("douglas_faqs.csv")